<a href="https://colab.research.google.com/github/jeanbaptiste-billaud/mlops_lecture/blob/main/MLflow_Dagshub.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MLflow and Dagshub integration

Welcome! In this notebook, you'll get hands-on experience with MLflow, DVC and Dagshub.
Here, you'll understand that MLOps is a complex ecosystem similar to a musical orchestra. Each tool has a role but must interact with other tools.

**Objective:** Conducting an end-to-end machine learning experiment: data management and versioning, experiment monitoring, and model deployment.

# Preliminary step
- **Create a fork from [the github repo](https://github.com/jeanbaptiste-billaud/MLflow_Dagshub) and linked it to dagshub, then execute bellow cells to setup the environment**
- **If you're on google colab: change execution type: Exécution/Modifier le type d'éxécution --> GPU-T4**

In [1]:
# installation de la cli github
%%bash
mkdir -p -m 755 /etc/apt/keyrings
wget -qO- https://cli.github.com/packages/githubcli-archive-keyring.gpg > /etc/apt/keyrings/githubcli-archive-keyring.gpg
chmod go+r /etc/apt/keyrings/githubcli-archive-keyring.gpg
echo "deb [arch=$(dpkg --print-architecture) signed-by=/etc/apt/keyrings/githubcli-archive-keyring.gpg] https://cli.github.com/packages stable main" > /etc/apt/sources.list.d/github-cli.list
apt-get update -qq && apt-get install -y -qq gh

(Reading database ... 127332 files and directories currently installed.)
Preparing to unpack .../archives/gh_2.103.0_amd64.deb ...
Unpacking gh (2.103.0) over (2.102.0) ...
Setting up gh (2.103.0) ...
Processing triggers for man-db (2.12.0-4build2) ...


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)


Open a terminal in your collab session and execute:

```
gh auth login --hostname github.com --git-protocol https --web
```



In [2]:
!gh auth status
!gh auth setup-git

github.com
  ✓ Logged in to github.com account jeanbaptiste-billaud (/root/.config/gh/hosts.yml)
  - Active account: true
  - Git operations protocol: https
  - Token: gho_************************************
  - Token scopes: 'gist', 'read:org', 'repo', 'workflow'


In [3]:
repo_owner='jeanbaptiste-billaud'
repo_name='MLflow_Dagshub'

!gh repo clone {repo_owner}/{repo_name}
%cd MLflow_Dagshub

!git config user.name {repo_owner}
!git config user.email "billaud.jeanbaptiste@gmail.com"

Cloning into 'MLflow_Dagshub'...
remote: Enumerating objects: 11, done.
remote: Counting objects: 100% (11/11), done.
remote: Compressing objects: 100% (7/7), done.
remote: Total 11 (delta 0), reused 8 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (11/11), done.
/content/MLflow_Dagshub


# Use the DagsHub client to get the dataset

In [ ]:
%pip install -q dagshub mlflow dvc
import dagshub, os
from dagshub.data_engine import datasources

dagshub.init(repo_owner='jeanbaptiste-billaud', repo_name='MLflow_Dagshub', mlflow=False)
ds = datasources.get(f'{repo_owner}/{repo_name}', 'Dataset_BUSI_with_GT')

res = ds.all()                         # requête : métadonnées seulement

local_dir = res.download_files(os.getcwd() + "/data")   # téléchargement réel
print(local_dir)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 3.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.1/51.1 kB 5.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 4.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.0/117.0 kB 7.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 274.0/274.0 kB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 116.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.1/4.1 MB 118.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 108.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 470.1/470.1 kB 42.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 270.4/270.4 kB 22.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.4/73.4 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.8/148.8 kB 17.1 MB/s eta 0:00:00


❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=0b0ba0fa-4199-4374-9bc7-093e5602b1dd&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=c2214fee2889521ae771cac92bc897d2dfd0ffc2a2316b08b97191315fa65cbf




Accessing as jeanbaptiste-billaud

Repository jeanbaptiste-billaud/MLflow_Dagshub initialized!

Output()

Output()

In [ ]:
# Setup  dvc with your dagshub credentials
%rm .dvc/config
!dvc remote add origin https://dagshub.com/{repo_owner}/{repo_name}.dvc
!dvc remote modify origin --local auth basic
!dvc remote modify origin --local user {repo_owner}
!dvc remote modify origin --local password <your pwd>
!dvc config core.autostage true
!dvc remote default origin
!git add .dvc/config

# Data Exploration

## Let's take a look to our datas

In [ ]:
RAW_DIR = local_dir / "Dataset_BUSI_with_GT"      # dossier issu de dvc get

for root, dirs, files in os.walk(RAW_DIR):
    print("Current directory:", root)
    print("Subdirectories:", dirs)
    print("Files:", files)
    print('----------------')

# Les données sont bien présentes

Current directory: /content/MLflow_Dagshub/data/Dataset_BUSI_with_GT
Subdirectories: ['benign', 'normal', 'malignant']
Files: []
----------------
Current directory: /content/MLflow_Dagshub/data/Dataset_BUSI_with_GT/benign
Subdirectories: []
Files: ['benign (99)_mask.png', 'benign (25)_mask.png', 'benign (410).png', 'benign (33).png', 'benign (112)_mask.png', 'benign (90).png', 'benign (290)_mask.png', 'benign (96).png', 'benign (180).png', 'benign (201).png', 'benign (184)_mask.png', 'benign (375).png', 'benign (373).png', 'benign (306).png', 'benign (362).png', 'benign (265).png', 'benign (323)_mask.png', 'benign (350).png', 'benign (420)_mask.png', 'benign (43).png', 'benign (61)_mask.png', 'benign (30)_mask.png', 'benign (3)_mask.png', 'benign (44)_mask.png', 'benign (140)_mask.png', 'benign (327)_mask.png', 'benign (423).png', 'benign (358).png', 'benign (27)_mask.png', 'benign (402).png', 'benign (429).png', 'benign (134)_mask.png', 'benign (151).png', 'benign (421).png', 'benign 

Our dataset is composed from 3 labels ['benign', 'normal', 'malignant'] and each sample have 2 images "main" and "mask". We'll not use "mask" images.

## Cleaning & formatting datas

In [ ]:
import re

OUT_DIR = local_dir / "processed"
CLASSES = ["benign", "malignant", "normal"]
SEED = 42
VAL_SIZE, TEST_SIZE = 0.15, 0.15

MASK_RE = re.compile(r"_mask(_\d+)?$", re.IGNORECASE)   # "_mask", "_mask_1", ...

In [ ]:
import pandas as pd

# Inventory and exclusion
rows = []
for label in CLASSES:
    for p in sorted((RAW_DIR / label).glob("*.png")):
        rows.append({
            "path": p.as_posix(),
            "label": label,
            "is_mask": bool(MASK_RE.search(p.stem)),
        })

all_files = pd.DataFrame(rows)
print(all_files.groupby(["label", "is_mask"]).size().unstack(fill_value=0))

df = all_files[~all_files["is_mask"]].drop(columns="is_mask").reset_index(drop=True)
print(f"\n{len(df)} images retenues sur {len(all_files)} fichiers")

is_mask    False  True 
label                  
benign       437    454
malignant    210    211
normal       133    133

780 images retenues sur 1578 fichiers


In [ ]:
import hashlib
from PIL import Image

# Duplicates
def file_md5(path, chunk=1 << 20):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(chunk), b""):
            h.update(block)
    return h.hexdigest()

def img_meta(path):
    with Image.open(path) as im:
        return im.width, im.height, im.mode

df["md5"] = df["path"].map(file_md5)
df[["width", "height", "mode"]] = df["path"].map(img_meta).tolist()
display(df.head())

dup_groups = df[df.duplicated("md5", keep=False)].sort_values("md5")
print(f"{dup_groups['md5'].nunique()} groupes de doublons, {len(dup_groups)} fichiers concernés")

# Doublons avec labels contradictoires : à signaler, on les exclut
conflicts = dup_groups.groupby("md5")["label"].nunique()
conflict_md5 = conflicts[conflicts > 1].index
print(f"{len(conflict_md5)} doublons avec labels contradictoires (exclus)")
display(dup_groups.head(10))

df = df[~df["md5"].isin(conflict_md5)]
df = df.drop_duplicates("md5", keep="first").reset_index(drop=True)
print(f"{len(df)} images uniques après dédoublonnage")

,path,label,md5,width,height,mode
0,/content/MLflow_Dagshub/data/Dataset_BUSI_with...,benign,96e5a08ef7b9c9429afac7aa75dfe0a6,562,471,RGB
1,/content/MLflow_Dagshub/data/Dataset_BUSI_with...,benign,c361932f412319342d95a68dd6229d50,683,585,RGB
2,/content/MLflow_Dagshub/data/Dataset_BUSI_with...,benign,c8ef9022b23dd1a8f1a3046b5b11ff41,323,473,RGB
3,/content/MLflow_Dagshub/data/Dataset_BUSI_with...,benign,6199b14831c04cebd43b7f568227c7d4,563,473,RGB
4,/content/MLflow_Dagshub/data/Dataset_BUSI_with...,benign,10c710780094c714b4c423385e21c13b,634,610,RGB


1 groupes de doublons, 2 fichiers concernés
1 doublons avec labels contradictoires (exclus)


,path,label,md5,width,height,mode
371,/content/MLflow_Dagshub/data/Dataset_BUSI_with...,benign,88f5ac307c1827b335d8b981c31262dc,550,462,RGB
488,/content/MLflow_Dagshub/data/Dataset_BUSI_with...,malignant,88f5ac307c1827b335d8b981c31262dc,550,462,RGB


778 images uniques après dédoublonnage


In [ ]:
from sklearn.model_selection import train_test_split

# laminated split
train_val, test = train_test_split(
    df, test_size=TEST_SIZE, stratify=df["label"], random_state=SEED
)
train, val = train_test_split(
    train_val, test_size=VAL_SIZE / (1 - TEST_SIZE),
    stratify=train_val["label"], random_state=SEED,
)

df["split"] = "train"
df.loc[val.index, "split"] = "val"
df.loc[test.index, "split"] = "test"

summary = pd.crosstab(df["label"], df["split"], margins=True)
display(summary)
display(pd.crosstab(df["label"], df["split"], normalize="columns").round(3))

split,test,train,val,All
label,,,,
benign,66,305,65,436
malignant,31,146,32,209
normal,20,93,20,133
All,117,544,117,778


split,test,train,val
label,,,
benign,0.564,0.561,0.556
malignant,0.265,0.268,0.274
normal,0.171,0.171,0.171


In [ ]:
# Export and track
OUT_DIR.mkdir(parents=True, exist_ok=True)
df["label_id"] = df["label"].map({c: i for i, c in enumerate(CLASSES)})
df = df[["path", "label", "label_id", "split", "width", "height", "mode", "md5"]]

df.to_csv(OUT_DIR / "index.csv", index=False)
dup_groups.to_csv(OUT_DIR / "duplicates_report.csv", index=False)
print(f"Index écrit : {OUT_DIR / 'index.csv'} ({len(df)} lignes)")

!dvc add {RAW_DIR} {OUT_DIR}
!git status

Index écrit : /content/MLflow_Dagshub/data/processed/index.csv (778 lignes)
⠋ Checking graph
Adding...:   0% 0/2 [00:00<?, ?file/s{'info': ' data/Dataset_BUSI_with_GT |'}]
!
          |0.00 [00:00,     ?file/s]
 23% 204/891 [00:00<00:00, 2.03kfile/s{'info': ''}]
 52% 459/891 [00:00<00:00, 2.34kfile/s{'info': ''}]
 79% 704/891 [00:00<00:00, 2.38kfile/s{'info': ''}]
 82% 943/1.16k [00:00<00:00, 2.08kfile/s{'info': ''}]
 73% 1.16k/1.58k [00:00<00:00, 2.07kfile/s{'info': ''}]
 88% 1.39k/1.58k [00:00<00:00, 2.16kfile/s{'info': ''}]
                                                       
!
  0% |          |0/? [00:00<?,    ?files/s]
                                           
Adding data/Dataset_BUSI_with_GT to cache:   0% 0.00/1.57k [00:00<?, ?file/s]
  0% 0.00/1.57k [00:00<?, ?file/s{'info': ''}]                               
 16% 245/1.57k [00:00<00:00, 2.45kfile/s{'info': ''}]
 33% 525/1.57k [00:00<00:00, 2.65kfile/s{'info': ''}]
 50% 791/1.57k [00:00<00:00, 2.46kfile/s{'info': ''}]
 66

In [ ]:
!git commit -m "add processed data"
!git push
!dvc push

# Sklearn & XGBoost

## Data formating

In [ ]:
IMG_SIZE = 224                       # torch (backbone pré-entraîné)
SK_SIZE = 64                         # sklearn : petite taille, vecteurs à plat
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
BATCH_SIZE = 32

splits = {s: df[df["split"] == s].reset_index(drop=True) for s in ["train", "val", "test"]}
{s: len(d) for s, d in splits.items()}

{'train': 544, 'val': 117, 'test': 117}

In [ ]:
import numpy as np

def load_sklearn_array(frame, size=SK_SIZE):
    X = np.empty((len(frame), size * size), dtype=np.float32)
    for i, path in enumerate(frame["path"]):
        with Image.open(path) as im:
            im = im.convert("L").resize((size, size), Image.BILINEAR)
        X[i] = np.asarray(im, dtype=np.float32).ravel() / 255.0
    return X, frame["label_id"].to_numpy()

X_train, y_train = load_sklearn_array(splits["train"])
X_val,   y_val   = load_sklearn_array(splits["val"])
X_test,  y_test  = load_sklearn_array(splits["test"])

print(X_train.shape, X_val.shape, X_test.shape)

# Cache pour ne pas relire les images à chaque run
np.savez_compressed(OUT_DIR / f"sklearn_{SK_SIZE}px.npz",
                    X_train=X_train, y_train=y_train,
                    X_val=X_val, y_val=y_val,
                    X_test=X_test, y_test=y_test)

(544, 4096) (117, 4096) (117, 4096)


## Train the sklearn model with mlflow tracking

In [ ]:
import mlflow

# Connexion à votre repo dagshub ppour utiliser mlflow
dagshub.init(repo_owner=repo_owner, repo_name=repo_name, mlflow=True)
mlflow.set_experiment("busi-classification")

INDEX_MD5 = hashlib.md5((OUT_DIR / "index.csv").read_bytes()).hexdigest()

def common_params(model_type):
    return {"model_type": model_type, "seed": SEED, "index_md5": INDEX_MD5,
            "n_train": len(splits["train"]), "n_val": len(splits["val"]),
            "n_test": len(splits["test"])}

def log_eval(y_true, y_pred, prefix):
    metrics = {f"{prefix}_accuracy": accuracy_score(y_true, y_pred),
               f"{prefix}_f1_macro": f1_score(y_true, y_pred, average="macro")}
    mlflow.log_metrics(metrics)
    return metrics

def log_report(y_true, y_pred):
    report = classification_report(y_true, y_pred, target_names=CLASSES)
    mlflow.log_text(report, "classification_report_test.txt")
    print(report)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from mlflow.models import infer_signature

with mlflow.start_run(run_name="sklearn_random_forest"):
    params = {"n_estimators": 200, "class_weight": "balanced", "img_size": SK_SIZE}
    mlflow.log_params({**common_params("sklearn_rf"), **params})

    t0 = time.time()
    rf = RandomForestClassifier(n_estimators=200, class_weight="balanced",
                                n_jobs=-1, random_state=SEED)
    rf.fit(X_train, y_train)
    mlflow.log_metric("train_time_s", time.time() - t0)

    log_eval(y_val, rf.predict(X_val), "val")
    y_pred = rf.predict(X_test)
    print(log_eval(y_test, y_pred, "test"))
    log_report(y_test, y_pred)

    mlflow.sklearn.log_model(rf, artifact_path="model",
                             signature=infer_signature(X_train[:5], rf.predict(X_train[:5])),
                             input_example=X_train[:1])

## Train the XGBoost model

In [ ]:
from xgboost import XGBClassifier
from sklearn.utils.class_weight import compute_sample_weight

with mlflow.start_run(run_name="xgboost"):
    params = {"n_estimators": 200, "max_depth": 6, "learning_rate": 0.1,
              "tree_method": "hist", "img_size": SK_SIZE}
    mlflow.log_params({**common_params("xgboost"), **params})

    t0 = time.time()
    xgb = XGBClassifier(**{k: v for k, v in params.items() if k != "img_size"},
                        random_state=SEED, n_jobs=-1)
    xgb.fit(X_train, y_train,
            sample_weight=compute_sample_weight("balanced", y_train),
            eval_set=[(X_val, y_val)], verbose=False)
    mlflow.log_metric("train_time_s", time.time() - t0)

    log_eval(y_val, xgb.predict(X_val), "val")
    y_pred = xgb.predict(X_test)
    print(log_eval(y_test, y_pred, "test"))
    log_report(y_test, y_pred)

    mlflow.xgboost.log_model(xgb, artifact_path="model",
                             signature=infer_signature(X_train[:5], xgb.predict(X_train[:5])),
                             input_example=X_train[:1])

# PyTorch

## Data formating

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T

train_tf = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.RandomHorizontalFlip(),
    T.RandomRotation(10),
    T.ColorJitter(brightness=0.2, contrast=0.2),
    T.ToTensor(),
    T.Normalize(MEAN, STD),
])

eval_tf = T.Compose([                # = la transfo qui ira dans BentoML
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(MEAN, STD),
])

class BusiDataset(Dataset):
    def __init__(self, frame, transform):
        self.paths = frame["path"].tolist()
        self.labels = frame["label_id"].tolist()
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        with Image.open(self.paths[idx]) as im:
            img = im.convert("RGB")
        return self.transform(img), self.labels[idx]

datasets_torch = {
    "train": BusiDataset(splits["train"], train_tf),
    "val":   BusiDataset(splits["val"], eval_tf),
    "test":  BusiDataset(splits["test"], eval_tf),
}

## Create the loader

In [ ]:
g = torch.Generator().manual_seed(SEED)

loaders = {
    s: DataLoader(ds, batch_size=BATCH_SIZE, shuffle=(s == "train"),
                  num_workers=2, generator=g if s == "train" else None)
    for s, ds in datasets_torch.items()
}

# Déséquilibre benign >> malignant > normal : poids pour CrossEntropyLoss
counts = splits["train"]["label_id"].value_counts().sort_index().to_numpy()
class_weights = torch.tensor(counts.sum() / (len(CLASSES) * counts), dtype=torch.float32)
print(dict(zip(CLASSES, class_weights.round(decimals=3).tolist())))

# Vérification d'un batch
xb, yb = next(iter(loaders["train"]))
print(xb.shape, xb.dtype, yb[:10])

## Load pretrained ResNet-18 and finetune it

In [ ]:
import torch.nn as nn
from torchvision.models import resnet18, ResNet18_Weights

device = "cuda" if torch.cuda.is_available() else "cpu"
EPOCHS, LR = 5, 1e-3

def predict_loader(model, loader):
    model.eval()
    preds, labels = [], []
    with torch.no_grad():
        for xb, yb in loader:
            preds.append(model(xb.to(device)).argmax(1).cpu())
            labels.append(yb)
    return torch.cat(labels).numpy(), torch.cat(preds).numpy()

with mlflow.start_run(run_name="torch_resnet18"):
    mlflow.log_params({**common_params("torch_resnet18"),
                       "epochs": EPOCHS, "lr": LR, "batch_size": BATCH_SIZE,
                       "img_size": IMG_SIZE, "mean": MEAN, "std": STD,
                       "frozen_backbone": True})

    torch.manual_seed(SEED)
    model = resnet18(weights=ResNet18_Weights.DEFAULT)
    for p in model.parameters():
        p.requires_grad = False
    model.fc = nn.Linear(model.fc.in_features, len(CLASSES))
    model = model.to(device)

    criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))
    optimizer = torch.optim.Adam(model.fc.parameters(), lr=LR)

    t0 = time.time()
    for epoch in range(EPOCHS):
        model.train()
        running = 0.0
        for xb, yb in loaders["train"]:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()
            running += loss.item() * len(xb)

        y_true, y_hat = predict_loader(model, loaders["val"])
        mlflow.log_metrics({"train_loss": running / len(datasets_torch["train"]),
                            "val_f1_macro_epoch": f1_score(y_true, y_hat, average="macro")},
                           step=epoch)
        print(f"epoch {epoch+1}/{EPOCHS} - loss {running / len(datasets_torch['train']):.3f}")
    mlflow.log_metric("train_time_s", time.time() - t0)

    log_eval(*predict_loader(model, loaders["val"]), "val")
    y_true, y_pred = predict_loader(model, loaders["test"])
    print(log_eval(y_true, y_pred, "test"))
    log_report(y_true, y_pred)

    model.cpu().eval()
    example = next(iter(loaders["test"]))[0][:1].numpy()
    mlflow.pytorch.log_model(model, artifact_path="model", input_example=example)

# Runs comparison

In [ ]:
runs = mlflow.search_runs(experiment_names=["busi-classification"],
                          order_by=["metrics.test_f1_macro DESC"])
display(runs[["tags.mlflow.runName", "metrics.test_accuracy",
              "metrics.test_f1_macro", "metrics.train_time_s"]])